# IPF walkthrough: raking via R's `surveysd` package

This notebook is the companion to `01_greg_walkthrough.ipynb`. It runs the same toy
survey through the IPF runner that sits in `paper-l0/benchmarking/runners/ipf_runner.R`.
The focus is the input and output formats IPF requires, which look very different from
GREG's because IPF is structurally not a generic-linear-system method.

IPF (Iterative Proportional Fitting, a.k.a. raking) adjusts unit weights so that the
weighted *counts* of a set of cross-tabulated categories match known population totals.
It works on categorical or indicator margins, not on arbitrary linear combinations of
unit attributes. This shows up as two concrete limitations in our benchmark:

1. Targets must be count-like. Dollar totals such as `district_A_income` cannot be
   calibrated with classical IPF and are dropped from this tier.
2. The input is a unit-record table with category columns plus a target table that
   lists each margin cell, not a sparse `(n_targets, n_units)` matrix.

In [1]:
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from toy_data import (
    HOUSEHOLDS,
    TARGETS,
    diagnostic_table,
    household_table_with_coefficients,
)

NOTEBOOK_DIR = Path().resolve()
IPF_RUNNER = NOTEBOOK_DIR.parent / "runners" / "ipf_runner.R"
assert IPF_RUNNER.exists(), IPF_RUNNER

## 1. The toy survey (same as the GREG walkthrough)

Ten households across two districts with adult/child breakdowns and household incomes.
The design weight is 100 for every household.

In [2]:
HOUSEHOLDS

,hh_id,district,n_adults,n_children,income,design_weight,hh_size
0,1,A,2,0,30000,100.0,2
1,2,A,2,2,60000,100.0,4
2,3,A,1,0,25000,100.0,1
3,4,A,2,1,45000,100.0,3
4,5,A,2,0,70000,100.0,2
5,6,B,2,3,40000,100.0,5
6,7,B,1,1,20000,100.0,2
7,8,B,2,1,80000,100.0,3
8,9,B,3,1,90000,100.0,4
9,10,B,1,0,15000,100.0,1


## 2. Target set — IPF-eligible subset

The GREG walkthrough calibrates eight targets, four per district. Of those, only
**six** are count-style and therefore runnable through classical IPF: adults and children
in each district, plus a household-level total in each district. The two `_income`
targets are dollar magnitudes and are dropped here.

For this walkthrough we stay with the person-level count targets only and use them to
build a single categorical margin over a `district_role` factor. Household counts would
require a separate household-level margin, which the `ipf_runner.R` supports, but we
skip them in the demo to keep the narrative concentrated on one margin construction.

In [3]:
ipf_target_specs = pd.DataFrame(
    [
        ("district_A_adults", "A_adult", 940.0),
        ("district_A_children", "A_child", 310.0),
        ("district_B_adults", "B_adult", 1000.0),
        ("district_B_children", "B_child", 600.0),
    ],
    columns=["target_name", "cell_label", "target_value"],
)
ipf_target_specs

,target_name,cell_label,target_value
0,district_A_adults,A_adult,940.0
1,district_A_children,A_child,310.0
2,district_B_adults,B_adult,1000.0
3,district_B_children,B_child,600.0


## 3. Input format 1/3 — a unit-record microdata table

IPF consumes one row per *observation* in the dimension of the constraints. Because our
targets are person-level counts, we expand the ten households into 27 person-level rows.
Every person in the same household keeps the same `unit_index` and the same
`base_weight`, so the fitted weights collapse cleanly back to one weight per household
after the IPF run.

The key column is `district_role`, a categorical that combines district and role. IPF
will read the target margin against this column.

In [4]:
rows = []
for _, r in HOUSEHOLDS.iterrows():
    hid = int(r["hh_id"])
    for _ in range(int(r["n_adults"])):
        rows.append({"hh_id": hid, "district": r["district"], "role": "adult"})
    for _ in range(int(r["n_children"])):
        rows.append({"hh_id": hid, "district": r["district"], "role": "child"})

persons = pd.DataFrame(rows)
persons["unit_index"] = persons["hh_id"] - 1  # 0-indexed household clone id
persons["household_id"] = persons["hh_id"]  # surveysd::ipf 'hid' column
persons["district_role"] = persons["district"] + "_" + persons["role"]
persons["base_weight"] = persons["hh_id"].map(
    dict(zip(HOUSEHOLDS["hh_id"], HOUSEHOLDS["design_weight"]))
)
persons.head(10)

,hh_id,district,role,unit_index,household_id,district_role,base_weight
0,1,A,adult,0,1,A_adult,100.0
1,1,A,adult,0,1,A_adult,100.0
2,2,A,adult,1,2,A_adult,100.0
3,2,A,adult,1,2,A_adult,100.0
4,2,A,child,1,2,A_child,100.0
5,2,A,child,1,2,A_child,100.0
6,3,A,adult,2,3,A_adult,100.0
7,4,A,adult,3,4,A_adult,100.0
8,4,A,adult,3,4,A_adult,100.0
9,4,A,child,3,4,A_child,100.0


The `unit_index` column matters: `ipf_runner.R` uses it to collapse per-person weights
back to per-unit weights. All persons sharing an `unit_index` must end the IPF run with
the same fitted weight; the runner enforces this and errors otherwise. (surveysd's
default `meanHH = TRUE` guarantees it.)

## 4. Input format 2/3 — target metadata

The IPF runner supports two encodings in `ipf_target_metadata.csv`:

- `numeric_total` rows, each pointing to an indicator column on the unit table. This is
  what `ipf_conversion.py` auto-generates from the saved calibration package.
- `categorical_margin` rows, one per cell of a margin table. Rows with the same
  `margin_id` are grouped into a single surveysd constraint.

For a clean textbook-style IPF demo we write the targets as four `categorical_margin`
rows over a single `district_role` margin.

In [5]:
ipf_targets = pd.DataFrame(
    {
        "margin_id": ["margin_role"] * 4,
        "scope": ["person"] * 4,
        "target_type": ["categorical_margin"] * 4,
        "variables": ["district_role"] * 4,
        "cell": [f"district_role={c}" for c in ipf_target_specs["cell_label"]],
        "target_value": ipf_target_specs["target_value"].astype(float),
        "target_name": ipf_target_specs["target_name"],  # carried for diagnostics only
    }
)
ipf_targets

,margin_id,scope,target_type,variables,cell,target_value,target_name
0,margin_role,person,categorical_margin,district_role,district_role=A_adult,940.0,district_A_adults
1,margin_role,person,categorical_margin,district_role,district_role=A_child,310.0,district_A_children
2,margin_role,person,categorical_margin,district_role,district_role=B_adult,1000.0,district_B_adults
3,margin_role,person,categorical_margin,district_role,district_role=B_child,600.0,district_B_children


## 5. Input format 3/3 — initial weights

Same `.npy` format as GREG: one weight per unit. Unit index here is the 0-indexed
household clone, so the vector has length 10 even though the unit table has 27 rows.

In [6]:
initial_weights = HOUSEHOLDS["design_weight"].to_numpy(dtype=np.float64)
initial_weights

array([100., 100., 100., 100., 100., 100., 100., 100., 100., 100.])

## 6. Write the bundle and invoke the R runner

The IPF runner signature is longer than GREG's because surveysd exposes more convergence
knobs. The ones that matter for a small problem like this one:

| argument | meaning | our choice |
| --- | --- | --- |
| `max_iter` | hard iteration cap | `5000` |
| `bound` | per-step weight change limit | `10.0` |
| `epsP` | person-constraint tolerance | `1e-4` |
| `epsH` | household-constraint tolerance | `1e-2` |
| `household_id_col` | hid column in the unit table | `household_id` |
| `weight_col` | base weight column | `base_weight` |

In [7]:
tmp = Path(tempfile.mkdtemp(prefix="ipf_toy_"))

persons_cols = [
    "unit_index",
    "household_id",
    "district",
    "role",
    "district_role",
    "base_weight",
]
persons[persons_cols].to_csv(tmp / "unit_metadata.csv", index=False)
ipf_targets.to_csv(tmp / "ipf_target_metadata.csv", index=False)
np.save(tmp / "initial_weights.npy", initial_weights)

out_csv = tmp / "fitted_weights.csv"
cmd = [
    "Rscript",
    str(IPF_RUNNER),
    str(tmp / "unit_metadata.csv"),
    str(tmp / "ipf_target_metadata.csv"),
    str(tmp / "initial_weights.npy"),
    str(out_csv),
    "5000",  # max_iter
    "10.0",  # bound
    "1e-4",  # epsP
    "1e-2",  # epsH
    "household_id",
    "base_weight",
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print("return code:", proc.returncode)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
sorted(p.name for p in tmp.iterdir())

return code: 0


['fitted_weights.csv',
 'initial_weights.npy',
 'ipf_target_metadata.csv',
 'unit_metadata.csv']

## 7. Output — per-row fitted weights, collapsed to per-unit

The IPF runner writes one row per person in `fitted_weights.csv`, keyed by `unit_index`.
Because surveysd enforces equal weights within each household (`meanHH = TRUE`), we can
take the first row per `unit_index` as the per-household fitted weight. This collapse
step is exactly what `benchmark_cli.py._run_ipf` does in production.

In [8]:
raw = pd.read_csv(out_csv)
raw["unit_index"] = raw["unit_index"].astype(int)

print("rows returned by IPF (one per person):", len(raw))
per_unit = (
    raw.groupby("unit_index", sort=True)["fitted_weight"]
    .first()
    .reindex(np.arange(len(HOUSEHOLDS), dtype=np.int64))
)
fitted_weights = per_unit.to_numpy(dtype=np.float64)

weights_view = pd.DataFrame(
    {
        "hh_id": HOUSEHOLDS["hh_id"],
        "district": HOUSEHOLDS["district"],
        "design_weight": initial_weights,
        "fitted_weight": fitted_weights,
        "ratio_to_design": fitted_weights / initial_weights,
    }
)
weights_view

rows returned by IPF (one per person): 27


,hh_id,district,design_weight,fitted_weight,ratio_to_design
0,1,A,100.0,105.238687,1.052387
1,2,A,100.0,103.096726,1.030967
2,3,A,100.0,105.238687,1.052387
3,4,A,100.0,103.806553,1.038066
4,5,A,100.0,105.238687,1.052387
5,6,B,100.0,90.806790,0.908068
6,7,B,100.0,98.013683,0.980137
7,8,B,100.0,111.210808,1.112108
8,9,B,100.0,118.408674,1.184087
9,10,B,100.0,142.671564,1.426716


## 8. Did we hit the person-count targets?

We diagnose only the four person-level count targets IPF was actually asked to fit.
The household-count and income targets that GREG received are outside IPF's natural
scope and are excluded from this diagnostic.

In [9]:
ipf_scored_targets = TARGETS[
    TARGETS["target_name"].isin(ipf_target_specs["target_name"])
].reset_index(drop=True)
diagnostic_table(
    household_table_with_coefficients(),
    ipf_scored_targets,
    fitted_weights,
).style.format(
    {
        "target_value": "{:,.1f}",
        "baseline_weighted_total": "{:,.1f}",
        "fitted_weighted_total": "{:,.1f}",
        "abs_rel_error": "{:.2e}",
    }
)

,target_name,target_value,baseline_weighted_total,fitted_weighted_total,abs_rel_error
0,district_A_adults,940.0,900.0,940.0,6.07e-09
1,district_A_children,310.0,300.0,310.0,1.84e-08
2,district_B_adults,"1,000.0",900.0,999.9,5.35e-05
3,district_B_children,600.0,600.0,600.1,8.92e-05


## 9. Known pathology — targets that IPF cannot accept

What classical IPF *cannot* do on this same toy survey:

- **`district_A_income` and `district_B_income`** — these are dollar totals, not cell
  counts. surveysd's raking expects margin structure, not arbitrary linear-combination
  targets. We drop them in this walkthrough.
- **Targets over unit-level attributes that cross household/person boundaries** — mixing
  these with person-level count margins needs care because `meanHH = TRUE` is what lets
  us collapse person weights back to one per household.

These are the core reasons the benchmark plan treats IPF as a *reference method on
count-style target subsets*, not a direct competitor to GREG or L0 on the full mixed
target set.

## 10. How the benchmark scaffold produces these inputs from raw target constraints

Sections 3-5 hand-built the unit table, the `district_role` category, and the margin rows directly. In production, those artifacts come out of `paper-l0/benchmarking/ipf_conversion.py`, which takes the **same `filtered_targets` DataFrame** that the GREG and L0 runners see — the manifest's `target_filters` controls target selection for all three methods — and internally drops anything IPF can't use. Two filters run inside the converter:

1. **Count check.** Keep only targets whose `variable` is a supported count (`person_count`, `household_count`). Dollar-total targets and tax-unit / SPM-unit counts stay in the shared sparse matrix for GREG and L0 but are dropped here. `tax_unit_count` and `spm_unit_count` need a separate unit table keyed by the respective entity; left out of this pass.
2. **Resolver check.** For each surviving target, map its stratum constraints to a categorical cell label using declared bucket schemas:

   | variable | schema | cell labels |
   | --- | --- | --- |
   | `age` | 18 five-year buckets | `0-4`, `5-9`, …, `85+` |
   | `adjusted_gross_income` (district) | 9 brackets | `(-inf,1)`, `[1,10k)`, …, `[500k,inf)` |
   | `adjusted_gross_income` (state) | 10 brackets | extends district up to `[1M,inf)` |
   | `eitc_child_count` | 4 discrete | `0`, `1`, `2`, `>2` |
   | any dollar var used with `> 0` | binary | `positive`, `non_positive` (positive is capped at $15T so the cell is a closed bucket) |
   | everything else with `==` | raw equality | the value itself |

   Targets whose constraints don't map to any declared schema are dropped.

The survivors are grouped into margin blocks by constraint signature, single-cell-per-geo blocks get their complement cell synthesized from the baseline weighted count of the complement predicate, and the blocks are emitted as `categorical_margin` rows. Both drops are non-fatal — counts land on `target_metadata.attrs['dropped_targets']` so the caller can report them.

### 10a. Input: a slice of `filtered_targets`

The converter takes exactly the same DataFrame the GREG and L0 paths take. Below is a tiny illustrative slice including a mix of count-style and non-count targets so the count check is visible. Four person-count targets span age buckets in two districts; two household-count targets ask for `snap > 0` in each district (single-cell-per-geo — these will need complement synthesis); the remaining two rows are non-count targets that the count check will drop.

In [10]:
# Mirror of what benchmark_manifest.filter_targets would produce for a small scope.
filtered_targets = pd.DataFrame(
    [
        {
            "target_id": 1,
            "stratum_id": 1,
            "variable": "person_count",
            "value": 210.0,
            "target_name": "pc_age_0-4_d601",
        },
        {
            "target_id": 2,
            "stratum_id": 2,
            "variable": "person_count",
            "value": 205.0,
            "target_name": "pc_age_5-9_d601",
        },
        {
            "target_id": 3,
            "stratum_id": 3,
            "variable": "person_count",
            "value": 200.0,
            "target_name": "pc_age_0-4_d602",
        },
        {
            "target_id": 4,
            "stratum_id": 4,
            "variable": "person_count",
            "value": 198.0,
            "target_name": "pc_age_5-9_d602",
        },
        {
            "target_id": 5,
            "stratum_id": 5,
            "variable": "household_count",
            "value": 80.0,
            "target_name": "hh_snap_pos_d601",
        },
        {
            "target_id": 6,
            "stratum_id": 6,
            "variable": "household_count",
            "value": 70.0,
            "target_name": "hh_snap_pos_d602",
        },
        # Non-count rows — dropped by the count check:
        {
            "target_id": 7,
            "stratum_id": 5,
            "variable": "adjusted_gross_income",
            "value": 1.2e7,
            "target_name": "agi_total_d601",
        },
        {
            "target_id": 8,
            "stratum_id": 6,
            "variable": "tax_unit_count",
            "value": 200.0,
            "target_name": "tax_units_d601",
        },
    ]
)
filtered_targets

,target_id,stratum_id,variable,value,target_name
0,1,1,person_count,210.0,pc_age_0-4_d601
1,2,2,person_count,205.0,pc_age_5-9_d601
2,3,3,person_count,200.0,pc_age_0-4_d602
3,4,4,person_count,198.0,pc_age_5-9_d602
4,5,5,household_count,80.0,hh_snap_pos_d601
5,6,6,household_count,70.0,hh_snap_pos_d602
6,7,5,adjusted_gross_income,12000000.0,agi_total_d601
7,8,6,tax_unit_count,200.0,tax_units_d601


Here are the stratum-constraint records for each of the six count-style targets (the remaining two rows have no bearing on the walkthrough because they'll be filtered out before resolution runs).

In [11]:
stratum_constraints = {
    1: [
        {"variable": "congressional_district_geoid", "operation": "==", "value": "601"},
        {"variable": "age", "operation": ">", "value": "-1"},
        {"variable": "age", "operation": "<", "value": "5"},
    ],
    2: [
        {"variable": "congressional_district_geoid", "operation": "==", "value": "601"},
        {"variable": "age", "operation": ">", "value": "4"},
        {"variable": "age", "operation": "<", "value": "10"},
    ],
    3: [
        {"variable": "congressional_district_geoid", "operation": "==", "value": "602"},
        {"variable": "age", "operation": ">", "value": "-1"},
        {"variable": "age", "operation": "<", "value": "5"},
    ],
    4: [
        {"variable": "congressional_district_geoid", "operation": "==", "value": "602"},
        {"variable": "age", "operation": ">", "value": "4"},
        {"variable": "age", "operation": "<", "value": "10"},
    ],
    5: [
        {"variable": "congressional_district_geoid", "operation": "==", "value": "601"},
        {"variable": "snap", "operation": ">", "value": "0"},
    ],
    6: [
        {"variable": "congressional_district_geoid", "operation": "==", "value": "602"},
        {"variable": "snap", "operation": ">", "value": "0"},
    ],
}
pd.DataFrame(
    [
        {
            "stratum_id": sid,
            "raw_constraints": "; ".join(
                f"{c['variable']}{c['operation']}{c['value']}" for c in cs
            ),
        }
        for sid, cs in stratum_constraints.items()
    ]
)

,stratum_id,raw_constraints
0,1,congressional_district_geoid==601; age>-1; age<5
1,2,congressional_district_geoid==601; age>4; age<10
2,3,congressional_district_geoid==602; age>-1; age<5
3,4,congressional_district_geoid==602; age>4; age<10
4,5,congressional_district_geoid==601; snap>0
5,6,congressional_district_geoid==602; snap>0


### 10b. The count check and the resolver check

The converter keeps only `person_count` and `household_count` targets, then walks each surviving target, groups its constraints by variable, and matches them against the declared schemas. The result is one `ResolvedTarget` per surviving target, with a `cell` tuple listing the `(derived_column, cell_label)` pairs that identify the cell.

In [12]:
import sys

sys.path.insert(0, str(NOTEBOOK_DIR.parent))
from ipf_conversion import (
    resolve_targets_for_testing,
    assemble_margins_for_testing,
    check_margin_consistency,
    split_target_metadata_by_margin,
    _SCOPE_BY_VARIABLE,
    _materialize_derived_columns,
    _should_synthesize_complement,
    _synthesize_complement_cells,
    _cell_assignments,
    _complement_cell_assignments,
)

# 1. Count check: keep only supported count-style target variables.
supported = set(_SCOPE_BY_VARIABLE.keys())
count_style = filtered_targets[
    filtered_targets["variable"].isin(supported)
].reset_index(drop=True)
dropped_non_count = filtered_targets[~filtered_targets["variable"].isin(supported)]
print(
    f"count check: kept {len(count_style)} / {len(filtered_targets)} targets; "
    f"dropped {len(dropped_non_count)}: {dropped_non_count['target_name'].tolist()}"
)

# 2. Resolver check: drop anything whose stratum constraints don't map to a declared cell.
resolved, unresolved = resolve_targets_for_testing(count_style, stratum_constraints)
print(f"resolver check: kept {len(resolved)}, dropped {len(unresolved)}")

pd.DataFrame(
    [
        {
            "target_name": r.target_name,
            "geo": f"{r.geo.variable}={r.geo.value}",
            "derived_cell": " & ".join(f"{c}={l}" for c, l in r.cell),
            "scope": r.scope,
            "target_value": r.target_value,
        }
        for r in resolved
    ]
)

count check: kept 6 / 8 targets; dropped 2: ['agi_total_d601', 'tax_units_d601']
resolver check: kept 6, dropped 0


,target_name,geo,derived_cell,scope,target_value
0,pc_age_0-4_d601,congressional_district_geoid=601,age_bracket=0-4,person,210.0
1,pc_age_5-9_d601,congressional_district_geoid=601,age_bracket=5-9,person,205.0
2,pc_age_0-4_d602,congressional_district_geoid=602,age_bracket=0-4,person,200.0
3,pc_age_5-9_d602,congressional_district_geoid=602,age_bracket=5-9,person,198.0
4,hh_snap_pos_d601,congressional_district_geoid=601,snap_positive=positive,household,80.0
5,hh_snap_pos_d602,congressional_district_geoid=602,snap_positive=positive,household,70.0


Note the two distinct derived columns that got produced:

- `age_bracket` (from the `age > X, age < Y` range constraints) → labels `0-4`, `5-9`.
- `snap_positive` (from the `snap > 0` single-threshold constraint) → label `positive`.

The congressional-district value passes through unchanged because geography variables
are already categorical.

### 10c. Group resolved targets into margin blocks

In [13]:
blocks = assemble_margins_for_testing(resolved)
blocks_view = pd.DataFrame(
    [
        {
            "margin_id": b.margin_id,
            "scope": b.scope,
            "cell_vars": " | ".join(b.cell_vars),
            "n_targets": len(b.targets),
            "n_geos": len({t.geo.value for t in b.targets}),
            "n_cells": len({t.cell for t in b.targets}),
        }
        for b in blocks
    ]
)
blocks_view

,margin_id,scope,cell_vars,n_targets,n_geos,n_cells
0,margin_0000,person,age_bracket | congressional_district_geoid,4,2,2
1,margin_0001,household,congressional_district_geoid | snap_positive,2,2,1


### 10d. Synthesize complement cells

The age-bracket block is already a complete 2×2 tile. The `snap > 0` block is a single
cell per district — IPF needs the complementary `snap_positive=non_positive` cell to
have a proper margin. The scaffold computes the complement count from baseline
microdata and emits it as a flagged `synthesized=True` row.

For the demo we need a unit table with the raw `snap` column so the complement can be
evaluated. We reuse this notebook's toy people and attach a synthetic `snap` column.

In [14]:
from toy_data import HOUSEHOLDS

rng = np.random.default_rng(0)
persons_for_conversion = pd.DataFrame(
    {
        "unit_index": np.arange(len(HOUSEHOLDS)),
        "household_id": np.arange(len(HOUSEHOLDS)),
        "base_weight": HOUSEHOLDS["design_weight"].astype(float).to_numpy(),
        "congressional_district_geoid": np.where(
            HOUSEHOLDS["district"] == "A", 601, 602
        ),
        "age": rng.integers(0, 10, size=len(HOUSEHOLDS)),
        "snap": np.array([0, 120, 0, 80, 0, 150, 0, 0, 200, 0], dtype=float),
    }
)

derived_needed = {
    cv
    for b in blocks
    for cv in b.cell_vars
    if cv not in ("congressional_district_geoid",)
}
raw_values = {
    "age": persons_for_conversion["age"].to_numpy(dtype=float),
    "snap": persons_for_conversion["snap"].to_numpy(dtype=float),
}
unit_table = _materialize_derived_columns(
    persons_for_conversion, derived_needed, raw_values
)

for b in blocks:
    if _should_synthesize_complement(b):
        _synthesize_complement_cells(b, unit_table, weight_column="base_weight")

snap_block = next(b for b in blocks if "snap_positive" in b.cell_vars)
pd.DataFrame(
    [
        {
            "target_name": t.target_name,
            "geo": t.geo.value,
            "cell": " & ".join(f"{c}={l}" for c, l in t.cell),
            "target_value": t.target_value,
            "synthesized": False,
        }
        for t in snap_block.targets
    ]
    + [
        {
            "target_name": f"margin_complement_{g}",
            "geo": g,
            "cell": " & ".join(f"{c}={l}" for c, l in cell),
            "target_value": val,
            "synthesized": True,
        }
        for g, cell, val in snap_block.synthesized_cells
    ]
)

,target_name,geo,cell,target_value,synthesized
0,hh_snap_pos_d601,601,snap_positive=positive,80.0,False
1,hh_snap_pos_d602,602,snap_positive=positive,70.0,False
2,margin_complement_601,601,snap_positive=non_positive,300.0,True
3,margin_complement_602,602,snap_positive=non_positive,300.0,True


### 10e. The emitted `ipf_target_metadata.csv` rows

With the cells resolved, margins assembled, and complements synthesized, each block
becomes a small block of `categorical_margin` rows. `ipf_runner.R` uses the
`margin_id` column to group rows into a single surveysd constraint per block.

In [15]:
out_rows = []
for b in blocks:
    vars_joined = "|".join(b.cell_vars)
    for t in b.targets:
        out_rows.append(
            {
                "margin_id": b.margin_id,
                "scope": b.scope,
                "target_type": "categorical_margin",
                "variables": vars_joined,
                "cell": "|".join(_cell_assignments(t, b)),
                "target_value": float(t.target_value),
                "target_name": t.target_name,
                "synthesized": False,
            }
        )
    for g, cell, val in b.synthesized_cells:
        out_rows.append(
            {
                "margin_id": b.margin_id,
                "scope": b.scope,
                "target_type": "categorical_margin",
                "variables": vars_joined,
                "cell": "|".join(_complement_cell_assignments(g, cell, b)),
                "target_value": float(val),
                "target_name": f"{b.margin_id}_complement_{g}",
                "synthesized": True,
            }
        )

emitted = pd.DataFrame(out_rows)
emitted

,margin_id,scope,target_type,variables,cell,target_value,target_name,synthesized
0,margin_0000,person,categorical_margin,age_bracket|congressional_district_geoid,age_bracket=0-4|congressional_district_geoid=601,210.0,pc_age_0-4_d601,False
1,margin_0000,person,categorical_margin,age_bracket|congressional_district_geoid,age_bracket=5-9|congressional_district_geoid=601,205.0,pc_age_5-9_d601,False
2,margin_0000,person,categorical_margin,age_bracket|congressional_district_geoid,age_bracket=0-4|congressional_district_geoid=602,200.0,pc_age_0-4_d602,False
3,margin_0000,person,categorical_margin,age_bracket|congressional_district_geoid,age_bracket=5-9|congressional_district_geoid=602,198.0,pc_age_5-9_d602,False
4,margin_0001,household,categorical_margin,congressional_district_geoid|snap_positive,congressional_district_geoid=601|snap_positive...,80.0,hh_snap_pos_d601,False
5,margin_0001,household,categorical_margin,congressional_district_geoid|snap_positive,congressional_district_geoid=602|snap_positive...,70.0,hh_snap_pos_d602,False
6,margin_0001,household,categorical_margin,congressional_district_geoid|snap_positive,congressional_district_geoid=601|snap_positive...,300.0,margin_0001_complement_601,True
7,margin_0001,household,categorical_margin,congressional_district_geoid|snap_positive,congressional_district_geoid=602|snap_positive...,300.0,margin_0001_complement_602,True


### 10f. Margin-consistency check and per-block splitting

`surveysd::ipf` rakes `conP` (person-scope) and `conH` (household-scope) constraints
independently, so it only requires margin-total agreement **within the same scope at
the same geography**. The check below runs on our two blocks and finds no issues,
because the age block is person-scope and the snap block is household-scope — they
don't need to match.

In [16]:
issues = check_margin_consistency(blocks)
print(f"{len(issues)} mismatched scope/geography combinations")
for iss in issues:
    totals = ", ".join(f"{m}={v:,.0f}" for m, v in iss["margin_totals"].items())
    print(
        f"  scope={iss['scope']} geo={iss['geo_value']} spread={iss['relative_spread']:.2%}  {totals}"
    )

0 mismatched scope/geography combinations


For a concrete example of a *real* mismatch, the `age × district` block and the
`agi × district × is_filer` block in the full calibration package are both
person-scope and share district geographies, but they represent different
populations (all persons vs. persons in filer tax units) and therefore imply
different totals per district. Feeding both into a single `surveysd::ipf` call fails
with `population totals for different constraints do not match`. The scaffold
handles this by running one margin block at a time via
`split_target_metadata_by_margin`, which returns a sub-frame per `margin_id`.

In [17]:
per_block = split_target_metadata_by_margin(emitted)
print("Target rows per block (suitable for independent IPF runs):")
for margin_id, sub in per_block.items():
    totals_by_geo = {}
    for _, r in sub.iterrows():
        geo_part = [
            a
            for a in r["cell"].split("|")
            if a.startswith("congressional_district_geoid=")
        ][0]
        totals_by_geo[geo_part] = totals_by_geo.get(geo_part, 0.0) + r["target_value"]
    totals_str = ", ".join(f"{g}->{v:,.0f}" for g, v in sorted(totals_by_geo.items()))
    print(f"  {margin_id}: {len(sub)} rows, per-geo totals: {totals_str}")

Target rows per block (suitable for independent IPF runs):
  margin_0000: 4 rows, per-geo totals: congressional_district_geoid=601->415, congressional_district_geoid=602->398
  margin_0001: 4 rows, per-geo totals: congressional_district_geoid=601->380, congressional_district_geoid=602->370


### 10g. How the same path runs against the real calibration package

In production, `benchmark_export.build_ipf_inputs` receives the `filtered_targets` slice computed by `benchmark_manifest.filter_targets` from the manifest's `target_filters`. For a three-district scope that filter returns roughly 80 targets across `person_count`, `household_count`, `tax_unit_count`, and a handful of dollar variables. The count check keeps the ~57 `person_count` + `household_count` rows; the resolver check keeps every one of those (they all fit the declared schemas). The survivors assemble into three margin blocks: 54-target `age × district`, 27-target `agi × district × is_filer` (if AGI targets are in scope), and 3-target `snap × district` with 3 synthesized complements. The dropped counts are available on `target_metadata.attrs['dropped_targets']` for reporting.

## Summary

Inputs to the IPF runner:

| Artifact | Format | Shape |
| --- | --- | --- |
| `unit_metadata.csv` | person-level CSV with category columns + `unit_index`, `household_id`, `base_weight` | `n_persons` rows |
| `ipf_target_metadata.csv` | CSV with one row per margin cell (`categorical_margin`) or per numeric total (`numeric_total`) | `n_cells` rows |
| `initial_weights.npy` | float64 array | `n_units` |

Output of the IPF runner:

| Artifact | Format | Shape |
| --- | --- | --- |
| `fitted_weights.csv` | CSV (`unit_index`, `fitted_weight`) | `n_persons` rows, collapsible to `n_units` |

Comparison to the GREG walkthrough:

| Aspect | GREG | IPF |
| --- | --- | --- |
| Primary input | sparse `(n_targets, n_units)` matrix | unit-record table with categorical columns |
| Target vocabulary | arbitrary linear functionals | count margins or indicator totals |
| Handles dollar-total targets? | yes | no (without an ad-hoc encoding) |
| Negative weights? | possible with `cal.linear` | no (multiplicative updates) |
| Collapse step needed? | no, weights already per-unit | yes, per-person weights → per-unit |